<a href="https://colab.research.google.com/github/Debadrita96/Personal-Projects/blob/main/rag_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%cd /content/sample_data/project


/content/sample_data/project


In [2]:
!pip install -q langchain langchain-openai langchain-community faiss-cpu pypdf python-dotenv tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.4/331.4 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [9]:
!pip install -q langchain-text-splitters

In [24]:
!pip install -q langchain-core langchain-community langchain-openai langchain

In [3]:
import langchain
import langchain_openai
import faiss
import pypdf

print("Libraries loaded successfully")

Libraries loaded successfully


In [4]:
import os

os.makedirs("data", exist_ok=True)
os.makedirs("vectorstore", exist_ok=True)

print("Folders created:", os.listdir("."))

Folders created: ['vectorstore', 'data']


In [5]:
docs = {
    "git_basics.txt": """
Git is a distributed version control system used to track changes in files.
A repository stores project files and their revision history.
Common commands include git init, git clone, git add, git commit, and git status.
""",
    "branches.txt": """
A branch is an independent line of development in Git.
Branches allow users to work on features without affecting the main branch.
Branches can later be merged into another branch.
""",
    "pull_requests.txt": """
A pull request is a request to review and merge changes from one branch into another.
Pull requests support discussion, code review, and approval before merging.
""",
    "merge_conflicts.txt": """
A merge conflict happens when Git cannot automatically reconcile differences between branches.
Conflicts must be resolved manually before the merge can be completed.
"""
}

for fname, content in docs.items():
    with open(os.path.join("data", fname), "w", encoding="utf-8") as f:
        f.write(content.strip())

print("Data folder files:", os.listdir("data"))

Data folder files: ['git_basics.txt', 'branches.txt', 'merge_conflicts.txt', 'pull_requests.txt']


In [16]:
import os

os.environ["OPENAI_API_KEY"] = "voc-2106197779175350455345469905267b15e46.33604292"
os.environ["OPENAI_API_BASE"] = "https://openai.vocareum.com/v1"

print("Vocareum API configured")

Vocareum API configured


In [7]:
from pathlib import Path

files_in_data = list(Path("data").glob("*"))
for f in files_in_data:
    print(f.name, f.suffix)

git_basics.txt .txt
branches.txt .txt
merge_conflicts.txt .txt
pull_requests.txt .txt


In [17]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

DATA_DIR = "data"
VECTORSTORE_DIR = "vectorstore"

def load_documents(data_dir):
    docs = []
    for file_path in Path(data_dir).glob("*"):
        if file_path.suffix.lower() == ".pdf":
            loader = PyPDFLoader(str(file_path))
            docs.extend(loader.load())
        elif file_path.suffix.lower() == ".txt":
            loader = TextLoader(str(file_path), encoding="utf-8")
            docs.extend(loader.load())
    return docs

documents = load_documents(DATA_DIR)
print("Number of loaded documents/pages:", len(documents))

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = text_splitter.split_documents(documents)
print("Number of chunks:", len(chunks))

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)
vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local(VECTORSTORE_DIR)

print(f"Vector store saved to: {VECTORSTORE_DIR}")

Number of loaded documents/pages: 4
Number of chunks: 4
Vector store saved to: vectorstore


In [32]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [34]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

db = FAISS.load_local(
    "vectorstore",
    embeddings,
    allow_dangerous_deserialization=True
)

retriever = db.as_retriever(search_kwargs={"k": 3})

print("Vectorstore loaded")

Vectorstore loaded


In [35]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

In [36]:
prompt = ChatPromptTemplate.from_template("""
You are a domain-specific assistant.

Use ONLY the context below to answer the question.

If the answer is not contained in the documents, reply exactly with:

"I don’t have enough information in the provided documents."

Context:
{context}

Question:
{question}
""")

In [37]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG pipeline ready")

RAG pipeline ready


In [38]:
print(rag_chain.invoke("What is a pull request?"))

A pull request is a request to review and merge changes from one branch into another. Pull requests support discussion, code review, and approval before merging.


In [39]:
print(rag_chain.invoke("Does Git automatically deploy code to production?"))

I don’t have enough information in the provided documents.


In [40]:
docs = retriever.invoke("What is a pull request?")

for d in docs:
    print("SOURCE:", d.metadata["source"])
    print(d.page_content)
    print()

SOURCE: data/pull_requests.txt
A pull request is a request to review and merge changes from one branch into another.
Pull requests support discussion, code review, and approval before merging.

SOURCE: data/git_basics.txt
Git is a distributed version control system used to track changes in files.
A repository stores project files and their revision history.
Common commands include git init, git clone, git add, git commit, and git status.

SOURCE: data/branches.txt
A branch is an independent line of development in Git.
Branches allow users to work on features without affecting the main branch.
Branches can later be merged into another branch.



In [41]:
%%writefile ingest.py
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

DATA_DIR = "data"
VECTORSTORE_DIR = "vectorstore"

def load_documents(data_dir):
    docs = []
    for file_path in Path(data_dir).glob("*"):
        if file_path.suffix.lower() == ".pdf":
            loader = PyPDFLoader(str(file_path))
            docs.extend(loader.load())
        elif file_path.suffix.lower() == ".txt":
            loader = TextLoader(str(file_path), encoding="utf-8")
            docs.extend(loader.load())
    return docs

def main():
    if "OPENAI_API_KEY" not in os.environ:
        raise ValueError("OPENAI_API_KEY not set.")

    documents = load_documents(DATA_DIR)

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=150
    )

    chunks = text_splitter.split_documents(documents)

    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

    vectorstore = FAISS.from_documents(chunks, embeddings)

    os.makedirs(VECTORSTORE_DIR, exist_ok=True)
    vectorstore.save_local(VECTORSTORE_DIR)

    print("Vectorstore created successfully")

if __name__ == "__main__":
    main()

Writing ingest.py


In [42]:
%%writefile chatbot.py
import os
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

VECTORSTORE_DIR = "vectorstore"

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def main():

    if "OPENAI_API_KEY" not in os.environ:
        raise ValueError("OPENAI_API_KEY not set.")

    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

    db = FAISS.load_local(
        VECTORSTORE_DIR,
        embeddings,
        allow_dangerous_deserialization=True
    )

    retriever = db.as_retriever(search_kwargs={"k": 3})

    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    prompt = ChatPromptTemplate.from_template("""
You are a domain support assistant.

Answer ONLY using the context below.

If the answer is not present in the documents, reply exactly with:

"I don’t have enough information in the provided documents."

Context:
{context}

Question:
{question}
""")

    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    print("RAG Assistant ready. Type 'exit' to quit.\n")

    while True:

        question = input("User: ")

        if question.lower() == "exit":
            break

        answer = rag_chain.invoke(question)

        print("Bot:", answer)
        print()

if __name__ == "__main__":
    main()

Writing chatbot.py


In [43]:
%%writefile sample_conversation.txt
User: What is a pull request?
Bot: A pull request is a request to review and merge changes from one branch into another.

User: Can it be reviewed before merging?
Bot: Yes. According to the documents, pull requests support discussion and code review before merging.

User: Does Git deploy code automatically?
Bot: I don’t have enough information in the provided documents.

Writing sample_conversation.txt


In [44]:
%%writefile README.md
# Week 15 Graded Mini Project – RAG Assistant

## Domain
Technology – Git Documentation Assistant

## Objective
Build a Retrieval-Augmented Generation (RAG) assistant that answers questions using only provided documents.

## Features
- Document ingestion and chunking
- OpenAI embeddings
- FAISS vector database
- Retrieval-Augmented generation
- Refusal when answer is not in documents
- Conversation-style interaction

## Data Sources
Example sources used:

- https://git-scm.com/docs
- https://docs.github.com/en/get-started

Documents were converted to TXT format and stored locally.

## Setup

Install dependencies:

pip install langchain langchain-community langchain-openai faiss-cpu

Set API key:

export OPENAI_API_KEY="your_api_key"

Run ingestion:

python ingest.py

Start chatbot:

python chatbot.py

## Example Questions

- What is a pull request?
- What is a Git branch?
- What are merge conflicts?
- Does Git deploy code automatically?

## Safety Behavior

If the answer is not in the documents the assistant responds with:

"I don’t have enough information in the provided documents."

## Files

- ingest.py
- chatbot.py
- README.md
- sample_conversation.txt
- Jupyter Notebook

Writing README.md


In [45]:
!ls

chatbot.py  data  ingest.py  README.md	sample_conversation.txt  vectorstore


In [46]:
%pwd

'/content/sample_data/project'

In [79]:
!zip -r rag_project.zip /content/sample_data/project

  adding: content/sample_data/project/ (stored 0%)
  adding: content/sample_data/project/ingest.py (deflated 57%)
  adding: content/sample_data/project/README.md (deflated 45%)
  adding: content/sample_data/project/vectorstore/ (stored 0%)
  adding: content/sample_data/project/vectorstore/index.faiss (deflated 16%)
  adding: content/sample_data/project/vectorstore/index.pkl (deflated 41%)
  adding: content/sample_data/project/chatbot.py (deflated 51%)
  adding: content/sample_data/project/.git/ (stored 0%)
  adding: content/sample_data/project/.git/ORIG_HEAD (stored 0%)
  adding: content/sample_data/project/.git/hooks/ (stored 0%)
  adding: content/sample_data/project/.git/hooks/push-to-checkout.sample (deflated 55%)
  adding: content/sample_data/project/.git/hooks/fsmonitor-watchman.sample (deflated 62%)
  adding: content/sample_data/project/.git/hooks/pre-merge-commit.sample (deflated 39%)
  adding: content/sample_data/project/.git/hooks/applypatch-msg.sample (deflated 42%)
  adding:

In [80]:
from google.colab import files
files.download("rag_project.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>